# FHIR R4 → OMOP CDM v5.5 via ViewDefinitions

The `fhir4ds.viewdef` OMOP mapping library flattens FHIR resources into
OMOP-shaped views. Concepts resolve through a ConceptMap RESOURCE join
(runtime join, `vocabulary_id + concept_code` keys); unmapped codes keep
their source values with concept_id 0. The composite runner assembles the
`omop.*` schema: child views → table-level UNION ALL → integer-key
projection (positive 63-bit hash) with concept-0 defaults.

In [1]:
# Install the unified package (PyPI).
# For a local wheel build instead:
#   %pip install -q --find-links "file:///abs/path/to/fhir4ds/dist" fhir4ds-v2==0.0.17
%pip install -q fhir4ds-v2==0.0.17


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
from pathlib import Path

import fhir4ds
from fhir4ds.viewdef.omop_runner import create_omop_views, OMOP_DIR

con = fhir4ds.create_connection()
con.execute("CREATE TABLE resources (resource JSON)")

### Stage the demo bundle

Two synthetic patients with the full spine (Encounter, Condition,
Procedure, MedicationRequest, Immunization, Observations) plus the
ConceptMap resources that carry the concept mappings
(element-per-resource form).

In [3]:
demo = OMOP_DIR / "demo" / "resources.ndjson"
for line in demo.read_text().splitlines():
    if line.strip():
        con.execute("INSERT INTO resources VALUES (?)", [line.strip()])
print(con.execute("SELECT count(*) FROM resources").fetchone()[0], "resources staged")

10 resources staged


### Build the omop.* schema

In [4]:
created = create_omop_views(con, source_table="resources")
print(sorted(created))

['__join_targets__', 'condition_occurrence', 'death', 'device_exposure', 'drug_exposure', 'measurement', 'observation', 'person', 'procedure_occurrence', 'visit_occurrence']


In [5]:
con.execute("SELECT * FROM omop.person").df()

,person_id,gender_concept_id,year_of_birth,month_of_birth,day_of_birth,birth_datetime,person_source_value,gender_source_value
0,4634691128028980087,8532,1930,01,01,1930-01-01,pat-002,female
1,6019492087770420870,8507,1974,12,25,1974-12-25,pat-001,male


In [6]:
con.execute("SELECT * FROM omop.condition_occurrence").df()

,condition_occurrence_id,person_id,condition_concept_id,condition_start_date,condition_start_datetime,condition_end_date,condition_end_datetime,condition_type_concept_id,condition_status_concept_id,condition_source_value,visit_occurrence_id
0,6539901264712631545,6019492087770420870,313217,2024-02-10,2024-02-10T08:30:00Z,None,None,43542353,0,38341003,5158441024566705761


In [7]:
con.execute("SELECT * FROM omop.drug_exposure").df()

,drug_exposure_id,person_id,drug_concept_id,drug_exposure_start_date,drug_exposure_start_datetime,drug_exposure_end_date,drug_exposure_end_datetime,drug_type_concept_id,sig,quantity_num,days_supply,drug_source_value,visit_occurrence_id,src
0,1740079870119879669,6019492087770420870,1128940,2024-03-02,2024-03-02T09:00:00Z,2024-03-02,2024-03-02T09:00:00Z,38000177,1 tablet daily,30.0,30,1049697,5158441024566705761,MedicationRequest
1,7129023373944063297,6019492087770420870,4056,2024-02-15,2024-02-15,2024-02-15,2024-02-15,38000180,None,NaN,<NA>,140,4564476756301768121,Immunization


### The value-type split

Quantity-valued Observations land in `measurement`; coded/string-valued
ones land in `observation` (the OMOP-on-FHIR project's Quantity rule,
adopted).

In [8]:
display(con.execute("SELECT * FROM omop.measurement").df())
display(con.execute("SELECT * FROM omop.observation").df())

,measurement_id,person_id,measurement_concept_id,measurement_date,measurement_datetime,measurement_type_concept_id,value_as_number,unit_source_value,range_low,range_high,measurement_source_value,value_source_value,visit_occurrence_id
0,2235477816070190481,6019492087770420870,3024929,2024-03-01,2024-03-01T08:00:00Z,32810,13.5,g/dL,12.0,17.5,718-7,13.5,5158441024566705761


,observation_id,person_id,observation_concept_id,observation_date,observation_datetime,observation_type_concept_id,value_as_string,observation_source_value,value_source_value,visit_occurrence_id
0,6216953706433833924,6019492087770420870,4076669,2024-03-01,2024-03-01T09:00:00Z,32810,Former smoker,72166-2,8517006,4564476756301768121


In [9]:
con.execute("SELECT * FROM omop.death").df()

,person_id,death_date,death_datetime,death_type_concept_id,cause_source_value
0,4634691128028980087,2024-06-01,2024-06-01T00:00:00Z,32810,2024-06-01T00:00:00Z
